In [7]:
# Assignment 10: Semantic Search using Word Embeddings

import numpy as np
import re
import os
from typing import cast
import gensim.downloader as api
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# STEP 1: LOAD PRETRAINED MODEL
# -----------------------------
def load_embedding_model() -> KeyedVectors:
    local_model_path = 'GoogleNews-vectors-negative300.bin'
    if os.path.exists(local_model_path):
        return KeyedVectors.load_word2vec_format(local_model_path, binary=True)

    print("GoogleNews file not found. Trying to download 'glove-wiki-gigaword-100'...")
    for attempt in range(3):
        try:
            return cast(KeyedVectors, api.load('glove-wiki-gigaword-100'))
        except Exception as exc:
            print(f"Download attempt {attempt + 1} failed: {exc}")

    raise RuntimeError(
        "Could not load embeddings. Place GoogleNews-vectors-negative300.bin in the workspace or check internet and retry."
    )

model = load_embedding_model()

# -----------------------------
# STEP 2: PREPROCESS FUNCTION
# -----------------------------
NORMALIZE_MAP = {
    'coding': 'programming',
    'code': 'programming',
    'coder': 'developer',
    'coders': 'developers',
    'laptop': 'laptops',
    'phone': 'smartphones',
    'phones': 'smartphones'
}

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    words = text.split()
    return [NORMALIZE_MAP.get(word, word) for word in words]

# -----------------------------
# STEP 3: SENTENCE EMBEDDING
# -----------------------------
def sentence_vector(sentence):
    words = preprocess(sentence)
    vectors = []

    for word in words:
        if word in model:
            vectors.append(model[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

# -----------------------------
# STEP 4: DATASET
# -----------------------------
documents = [
    "Best lightweight laptops for software engineering students",
    "High performance coding laptops with long battery life",
    "Affordable ultrabooks for Python and web development",
    "Top monitors and keyboards for programming setups",
    "Beginner roadmap for machine learning projects",
    "NLP interview preparation with practical examples",
    "How to optimize laptop RAM and SSD for faster builds",
    "Budget smartphones with strong camera performance",
    "Healthy weekly meal prep ideas for students",
    "Cloud deployment guide for full stack applications"
]

doc_vectors = [sentence_vector(doc) for doc in documents]

# -----------------------------
# STEP 5: SEARCH FUNCTION
# -----------------------------
def semantic_search(query, top_k=1):
    query_vec = sentence_vector(query)

    similarities = []
    for vec in doc_vectors:
        sim = cosine_similarity(
            np.asarray(query_vec).reshape(1, -1),
            np.asarray(vec).reshape(1, -1)
        )[0][0]
        similarities.append(sim)

    # sort results
    ranked = sorted(
        list(zip(documents, similarities)),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:top_k]

# -----------------------------
# STEP 6: TEST
# -----------------------------
query = "best laptop for coding"
results = semantic_search(query, top_k=3)

print("Query:", query)
print("\nTop Results:")
for res, score in results:
    print(f"{res}  --> similarity: {score:.4f}")

GoogleNews file not found. Trying to download 'glove-wiki-gigaword-100'...
[==================================================] 100.0% 128.1/128.1MB downloaded
Query: best laptop for coding

Top Results:
Best lightweight laptops for software engineering students  --> similarity: 0.8994
Top monitors and keyboards for programming setups  --> similarity: 0.8877
High performance coding laptops with long battery life  --> similarity: 0.8848
